# NYU Torch 上训练 CIFAR-10 变体 ResNet（≤5M 参数）

本 notebook 是本项目在 **NYU Torch** 上跑训练的完整操作手册，同时也是一个可以在 **Open OnDemand / Jupyter** 里逐步执行的 notebook。

前置说明（来自官方文档 `services.rt.nyu.edu/docs/hpc/`，已核实）：
- 投作业必须有 active allocation，且 `sbatch` 必须带 `--account=`（用 `my_slurm_accounts` 查询）
- **算力节点没有外网**，数据必须提前上传
- 不要手写 `--partition`，资源由 QoS 分配
- GPU util 过低会被自动 cancel：gl\*/ga\*/gr\* 50%、gh\* 60%、默认 10%
- 用户 GPU 配额：48 小时内最多 24 张卡

本机（Mac）现在解析不了 `login.torch.hpc.nyu.edu`，**需要先连 NYU VPN**，之后下面命令可直接跑。


## 0. 连通性自测

在本机终端（不是 notebook 里）执行；返回主机名 + `torch_pr_xxx` 即通。

In [ ]:
ssh -o BatchMode=yes -o ConnectTimeout=15 torch "hostname; my_slurm_accounts"
# 期望：log-1 / log-2 或 cm-xxx，以及 torch_pr_xxxxxx_xxxxx

## 1. 建目录并上传代码 + 数据（本机执行）

数据约 150 MB；算力节点无外网，所以必须随代码一起上去。

In [ ]:
# 远端正目录
ssh torch 'mkdir -p /scratch/$USER/dl26fall/{data,code,runs,submissions}'

# 代码（排除本机 venv / 缓存 / 图片）
rsync -av --exclude '.venv' --exclude 'runs' --exclude '.cache' \
      --exclude 'train_images' --exclude 'test_images' --exclude '.git' \
      ./ torch:/scratch/\$USER/dl26fall/code/

# 数据（一次性）
rsync -av train_images test_images train_labels.csv test.csv sample_submission.csv \
      torch:/scratch/\$USER/dl26fall/data/

In [ ]:
# 上传后校验张数：应输出 50000 / 2000
ssh torch "cd /scratch/\$USER/dl26fall/data && ls train_images | wc -l && ls test_images | wc -l"

## 2. 准备 PyTorch 环境（overlay + SIF）

官方推荐 Apptainer + conda overlay：

- SIF：`/share/apps/images/cuda12.1.1-cudnn8.9.0-devel-ubuntu22.04.2.sif`
- overlay：`/scratch/$USER/dl26fall/pytorch.ext3`
- 运行时：`singularity exec --nv --overlay <overlay>:ro <sif> /bin/bash -c "source /ext3/env.sh; python ..."`

在**交互节点**里建（不要用登录节点）：


In [ ]:
srun --pty -c 2 --mem=8GB --time=1:00:00 --account=torch_pr_XXXXXX_XXXXX /bin/bash
# —— 进入节点后 ——
singularity exec --overlay /scratch/$USER/dl26fall/pytorch.ext3:rw \
    /share/apps/images/cuda12.1.1-cudnn8.9.0-devel-ubuntu22.04.2.sif /bin/bash
# —— 进入容器后 ——
source /ext3/env.sh
pip install --no-cache-dir torch torchvision numpy pillow
python -c "import torch;print(torch.__version__, torch.cuda.is_available())"
exit; exit

## 3. 提交训练作业

模板已在仓库 `slurm/train.sbatch`；提交：

In [ ]:
ssh torch 'cd /scratch/$USER/dl26fall/code && sbatch slurm/train.sbatch'
# 观察队列与日志
ssh torch 'squeue -u $USER'
ssh torch 'cd /scratch/$USER/dl26fall/code && tail -f runs/hpc_*.log'

### 关键 sbatch 头（照抄即可，替换 account）

```
#SBATCH --gres=gpu:1
#SBATCH --cpus-per-task=8
#SBATCH --mem=32GB
#SBATCH --time=06:00:00
#SBATCH --account=torch_pr_XXXXXX_XXXXX
#SBATCH --comment="preemption=yes;requeue=true"
```

- `preemption=yes` → 可借用别人空闲 GPU，被抢占后自动重新排队（30 分钟内不会被抢）
- 想指定卡型：`--constraint='h200|l40s'`
- 想看真实卡型：`sinfo -o "%N %G %P %t"`


## 4. 断点续训

`sbatch` 被抢占后会自动重投，脚本检测到 `runs/wrn22x4_s0/last.pt` 就带 `--resume`。
本机版 `train.py` 每 epoch 存 `last.pt`；HPC 上 300 epoch ≈ 3–4 小时，`--time=06:00:00` 一次跑完最省心。

## 5. 取回产物并提交 Kaggle（本机执行）

HPC 上不装 kaggle 凭据，拉回来提交更干净。

In [ ]:
rsync -av torch:/scratch/\$USER/dl26fall/runs/wrn22x4_s0/final.pt ./runs/hpc/
.venv/bin/python src/predict.py --ckpt runs/hpc/final.pt --out submissions/hpc_tta.csv
kaggle competitions submit -c cifar-10-res-net-architecture-challenge-dl \
    -f submissions/hpc_tta.csv -m "hpc preact22x4 300ep tta" 

## 6. 验收清单

- [ ] `ssh torch "hostname"` 通
- [ ] `my_slurm_accounts` 有 account
- [ ] `/scratch/$USER/dl26fall/data` 下 50000 + 2000 张图
- [ ] 容器内 `torch.cuda.is_available() == True`
- [ ] `squeue` 看到作业 RUNNING
- [ ] 日志出现 `[ep 001/300]`
- [ ] 完成后 `final.pt` 存在且 `best_val` 打印出来
